In [0]:
# Install or update the bhom-schema package
%pip install --upgrade --pre --no-cache-dir --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ bhom-schema

Looking in indexes: https://test.pypi.org/simple/, https://pypi.org/simple/
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import bhom_schema
print(bhom_schema.__version__)

10.0.0a2611051628


In [0]:
from bhom_schema.FixtureAlpha_oM.LineLike import LineLike
from bhom_schema.FixtureAlpha_oM.PointLike import PointLike
from bhom_schema.tables import *
from bhom_schema.tables import describe

In [0]:
# Get SQL column type for primitive types
print(column_type(float))
print(column_type(list[str]))

# This will fail because PointLike is a custom type
# column_type(PointLike)

DOUBLE
ARRAY<STRING>


In [0]:
# Show required columns for a new table
for column in describe(PointLike):
    print(column.name, column.type, column.decision)

_id STRING key
_bhomVersion STRING primitive
X DOUBLE primitive
Y DOUBLE primitive
Z DOUBLE primitive


In [0]:
# Show required columns for a new table
for column in describe(LineLike):
    print(column.name, column.type, column.decision)

_id STRING key
_bhomVersion STRING primitive
Start None undecided
End None undecided
Infinite BOOLEAN primitive
Label STRING primitive


In [0]:

POINTS = "student_area.vcl.PointLike"
LINES = "student_area.vcl.LineLike"
REFS = {"Start": Ref(POINTS), "End": Ref(POINTS)}

# Generate SQL from the type to create new tables
print(create_table_sql(PointLike, POINTS))
print()
print(create_table_sql(LineLike, LINES, fields=REFS))
print()
print(create_table_sql(LineLike, LINES + "Struct", fields={"Start": Struct(), "End": Struct()}))

# Demonstrate UnmappedType failure
# create_table_sql(LineLike, LINES)

CREATE TABLE IF NOT EXISTS `student_area`.`vcl`.`PointLike` (
  `_id` STRING NOT NULL COMMENT 'Key of the row.',
  `_bhomVersion` STRING COMMENT 'Optional version of BHoM used as part of automatic versioning and schema upgrades.',
  `X` DOUBLE,
  `Y` DOUBLE,
  `Z` DOUBLE
)
USING DELTA
TBLPROPERTIES ('bhom.type' = 'BH.oM.FixtureAlpha.PointLike', 'bhom.schema_version' = '10.0.0a2611051628')

CREATE TABLE IF NOT EXISTS `student_area`.`vcl`.`LineLike` (
  `_id` STRING NOT NULL COMMENT 'Key of the row.',
  `_bhomVersion` STRING COMMENT 'Optional version of BHoM used as part of automatic versioning and schema upgrades.',
  `Start_id` STRING,
  `End_id` STRING,
  `Infinite` BOOLEAN COMMENT 'Extends the line to infinity.',
  `Label` STRING
)
USING DELTA
COMMENT 'A straight segment between two points.'
TBLPROPERTIES ('bhom.type' = 'BH.oM.FixtureAlpha.LineLike', 'bhom.schema_version' = '10.0.0a2611051628')

CREATE TABLE IF NOT EXISTS `student_area`.`vcl`.`LineLikeStruct` (
  `_id` STRING NOT NUL

In [0]:
# Create new tables
spark.sql(create_table_sql(LineLike, POINTS, fields=REFS))
spark.sql(create_table_sql(LineLike, LINES, fields=REFS))

spark.sql(f"DESCRIBE TABLE {POINTS}").show(truncate=False)
spark.sql(f"DESCRIBE TABLE {LINES}").show(truncate=False)

+------------------------+---------+----------------------------------------------------------------------------------+
|col_name                |data_type|comment                                                                           |
+------------------------+---------+----------------------------------------------------------------------------------+
|_id                     |string   |Key of the row.                                                                   |
|_bhomVersion            |string   |Optional version of BHoM used as part of automatic versioning and schema upgrades.|
|X                       |double   |NULL                                                                              |
|Y                       |double   |NULL                                                                              |
|Z                       |double   |NULL                                                                              |
|# Clustering Information|         |    

In [0]:
# Add test data
spark.sql(f"""
    INSERT INTO {POINTS} (`_id`, `X`, `Y`, `Z`) VALUES
    ('p1', 1, 0, 0), ('p2', 2, 0, 0), ('p3', 3, 4, 0)
""")
spark.sql(f"""
    INSERT INTO {LINES} (`_id`, `Start_id`, `End_id`, `Infinite`, `Label`) VALUES
    ('l1', 'p1', 'p2', false, 'axis'), ('l2', 'p2', 'p3', true, NULL)
""")
print(spark.table(POINTS).count(), spark.table(POINTS).count())
print(spark.table(POINTS).count(), spark.table(LINES).count())
spark.table(POINTS).show()
spark.table(LINES).show()

21 21
21 18
+---+------------+---+---+---+
|_id|_bhomVersion|  X|  Y|  Z|
+---+------------+---+---+---+
| p1|        NULL|1.0|0.0|0.0|
| p2|        NULL|2.0|0.0|0.0|
| p3|        NULL|3.0|4.0|0.0|
| p1|        NULL|1.0|0.0|0.0|
| p2|        NULL|2.0|0.0|0.0|
| p3|        NULL|3.0|4.0|0.0|
| p1|        NULL|1.0|0.0|0.0|
| p2|        NULL|2.0|0.0|0.0|
| p3|        NULL|3.0|4.0|0.0|
| p1|        NULL|1.0|0.0|0.0|
| p2|        NULL|2.0|0.0|0.0|
| p3|        NULL|3.0|4.0|0.0|
| p1|        NULL|1.0|0.0|0.0|
| p2|        NULL|2.0|0.0|0.0|
| p3|        NULL|3.0|4.0|0.0|
| p1|        NULL|1.0|0.0|0.0|
| p2|        NULL|2.0|0.0|0.0|
| p3|        NULL|3.0|4.0|0.0|
| p1|        NULL|1.0|0.0|0.0|
| p2|        NULL|2.0|0.0|0.0|
+---+------------+---+---+---+
only showing top 20 rows
+--------------------+------------+--------+------+--------+-----+
|                 _id|_bhomVersion|Start_id|End_id|Infinite|Label|
+--------------------+------------+--------+------+--------+-----+
|8b449a6e-7492-40c

In [0]:
# Create line, convert to table row, add to table
line = LineLike(Start=PointLike(X=5), End=PointLike(X=6), Label="new")
row = to_row(line, fields=REFS, ids={"Start": "p4", "End": "p5"})
spark.createDataFrame([row], schema=ddl_schema(LineLike, fields=REFS)).write.mode("append").saveAsTable(POINTS)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7290439453105425>, line 4
      2 line = LineLike(Start=PointLike(X=5), End=PointLike(X=6), Label="new")
      3 row = to_row(line, fields=REFS, ids={"Start": "p4", "End": "p5"})
----> 4 spark.createDataFrame([row], schema=ddl_schema(LineLike, fields=REFS)).write.mode("append").saveAsTable(POINTS)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:828, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    826 self._write.table_name = name
    827 self._write.table_save_method = "save_as_table"
--> 828 _, _, ei = self._spark.client.execute_command(
    829     self._write.command(self._spark.client), self._write.observations
    830 )
    831 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1812, in

In [0]:
print(spark.table(POINTS).count())
print(spark.table(LINES).count())

# read row and convert to object of custom type
stored = spark.table(POINTS).where("`_id` = 'l1'").collect()[0].asDict()
stored = spark.table(LINES).where("`_id` = 'l1'").collect()[0].asDict()
rebuilt = from_row(LineLike, stored, fields=REFS, held={"Start": points[stored["Start_id"]], "End": points[stored["End_id"]]})
line_rows = [r.asDict() for r in spark.table(POINTS).collect()]

21


In [0]:
line_rows = [r.asDict() for r in spark.table(LINES).collect()]
point_rows = [r.asDict() for r in spark.table(POINTS).collect()]

# Convert multiple rows to list of objects
points = rows_to_objects(PointLike, point_rows, keyed=True)
lines = rows_to_objects(LineLike, line_rows, fields=REFS, held={"Start": points, "End": points})
print(len(lines))
for line in lines:
    print(line.Label, line.Start.X, line.End.X)

objects, problems = rows_to_objects(PointLike, [{"_id": "a", "X": 1.0}, {"_id": "b", "X": "bad"}], errors="collect")
print(len(objects), problems[0].index, problems[0].key)

18
new 0.0 0.0
new 0.0 0.0
new 0.0 0.0
new 0.0 0.0
axis 1.0 2.0
None 2.0 3.0
axis 1.0 2.0
None 2.0 3.0
axis 1.0 2.0
None 2.0 3.0
axis 1.0 2.0
None 2.0 3.0
axis 1.0 2.0
None 2.0 3.0
axis 1.0 2.0
None 2.0 3.0
axis 1.0 2.0
None 2.0 3.0
1 1 b


/local_disk0/.ephemeral_nfs/envs/pythonEnv-b457ad38-8f80-4338-9af0-956adff9502a/lib/python3.12/site-packages/bhom_schema/tables/rows.py:308: UserWarning: Start is a Ref to the row p4, but held has no object for it, so it keeps its default
  found, problems = read_many(model, rows, fields, key, held, errors)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-b457ad38-8f80-4338-9af0-956adff9502a/lib/python3.12/site-packages/bhom_schema/tables/rows.py:308: UserWarning: End is a Ref to the row p5, but held has no object for it, so it keeps its default
  found, problems = read_many(model, rows, fields, key, held, errors)


In [0]:
from bhom_schema.FixtureAlpha_oM.Beam import Beam

BEAMS = "student_area.vcl.Beam"
BEAM_FIELDS = {**BASE_SKIPPED, "Start": Struct()}

# a. A wrong value set in Python is flagged where it is set
Beam(Start=PointLike(), Support="Banana")
beam = Beam(Start=PointLike())
beam.Support = "Fixd"

# b. The flags enum takes names combined with ", ", and nothing else
print(Beam(Start=PointLike(), Flags="A, B").Flags)
Beam(Start=PointLike(), Flags="A, Z")

# c. A wrong value written by hand is refused by the table
spark.sql(f"DROP TABLE IF EXISTS {BEAMS}")
spark.sql(create_table_sql(Beam, BEAMS, fields=BEAM_FIELDS))

for statement in enum_constraints_sql(Beam, BEAMS, BEAM_FIELDS):
    print(statement)
    spark.sql(statement)

spark.sql(f"INSERT INTO {BEAMS} (`BHoM_Guid`, `Support`, `Flags`) VALUES ('b1', 'Fixed', 'A, B'), ('b2', NULL, NULL)")
print(spark.table(BEAMS).count())

spark.sql(f"INSERT INTO {BEAMS} (`BHoM_Guid`, `Support`) VALUES ('b3', 'Bogus')")
spark.sql(f"INSERT INTO {BEAMS} (`BHoM_Guid`, `Flags`) VALUES ('b4', 'A, Z')")


---------------------------------------------------------------------------
ValidationError                           Traceback (most recent call last)
File <command-7290439453105557>, line 7
      4 BEAM_FIELDS = {**BASE_SKIPPED, "Start": Struct()}
      6 # a. A wrong value set in Python is flagged where it is set
----> 7 Beam(Start=PointLike(), Support="Banana")
      8 beam = Beam(Start=PointLike())
      9 beam.Support = "Fixd"

File /databricks/python/lib/python3.12/site-packages/pydantic/main.py:263, in BaseModel.__init__(self, **data)
    261 # `__tracebackhide__` tells pytest and some other tools to omit this function from tracebacks
    262 __tracebackhide__ = True
--> 263 validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
    264 if self is not validated_self:
    265     warnings.warn(
    266         'A custom validator is returning a value other than `self`.\n'
    267         "Returning anything other than `self` from a top level model